# PPE Compliance Checker - Training (Google Colab)

**ITAI 1378 midterm.** This notebook covers the full workflow:
download the dataset, train YOLO11, evaluate, run the compliance demo, and zip the results.

## How to run
1. **Runtime > Change runtime type > T4 GPU > Save.**
2. **Runtime > Run all.**
3. The **first run installs Ultralytics and auto-restarts the runtime - this is normal.** When it reconnects, do **Runtime > Run all** once more. It then trains, evaluates, runs the compliance demo, and downloads `ppe_results.zip`.

Training takes roughly 15-25 minutes on a free T4 GPU.

## 1 - Install (first run restarts the runtime - expected)

In [ ]:
# Colab preloads torch; installing a package that touches torch needs a runtime
# restart afterward, or `import torch` throws a circular-import error. This cell
# installs Ultralytics once, restarts, then skips on every later run.
import importlib.util, subprocess, sys, os
if importlib.util.find_spec('ultralytics') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'])
    print('Installed Ultralytics. Restarting runtime - when it reconnects, do Runtime > Run all again.')
    os.kill(os.getpid(), 9)   # auto-restart so torch imports cleanly
else:
    print('Ultralytics already installed - good to go.')

## 2 - GPU check

In [ ]:
import torch, ultralytics
ultralytics.checks()
print('GPU available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Do: Runtime > Change runtime type > T4 GPU, then Run all again.'

## 3 - Train YOLO11 on the Construction-PPE dataset
The dataset (1,132 train / 143 val / 141 test images, 178 MB) auto-downloads on the first run. Classes: `helmet, gloves, vest, boots, goggles, none, Person, no_helmet, no_goggle, no_gloves, no_boots`.

*For a faster first pass set `MODEL='yolo11n.pt'` and `EPOCHS=20`.*

In [ ]:
from ultralytics import YOLO

MODEL  = 'yolo11s.pt'   # 's' = good accuracy/speed; 'yolo11n.pt' = faster first pass
EPOCHS = 40
IMGSZ  = 640

model = YOLO(MODEL)
results = model.train(
    data='construction-ppe.yaml',
    epochs=EPOCHS, imgsz=IMGSZ, batch=16,
    patience=15, name='ppe', plots=True,
)
RUN_DIR = str(results.save_dir)
print('Training outputs saved to:', RUN_DIR)

## 4 - Evaluate: real metrics on the held-out set

In [ ]:
metrics = model.val()
print(f'mAP@50      : {metrics.box.map50:.3f}')
print(f'mAP@50-95   : {metrics.box.map:.3f}')
print(f'mean precision: {metrics.box.mp:.3f}   mean recall: {metrics.box.mr:.3f}')
for i, name in model.names.items():
    try:
        print(f'  {name:12s} mAP@50 = {metrics.box.maps[i]:.3f}')
    except Exception:
        pass

## 5 - Training curves + confusion matrix

In [ ]:
from IPython.display import Image, display
import os
for f in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    p = os.path.join(RUN_DIR, f)
    if os.path.exists(p):
        print(f); display(Image(filename=p, width=780))

## 6 - Compliance demo: label each worker Compliant / Non-compliant
Rule per person: **non-compliant** if a `no_helmet` or `no_goggle` box overlaps them, or if no `vest` overlaps them; **compliant** otherwise.

In [ ]:
import cv2, glob, os
from ultralytics.utils import SETTINGS

best = os.path.join(RUN_DIR, 'weights', 'best.pt')
det  = YOLO(best)
names = det.names

def center_in(box, person):
    cx, cy = (box[0]+box[2])/2, (box[1]+box[3])/2
    return person[0] <= cx <= person[2] and person[1] <= cy <= person[3]

def annotate(img_path, conf=0.35):
    img = cv2.imread(img_path)
    r = det(img_path, conf=conf, verbose=False)[0]
    dets = [(names[int(c)], list(map(float, b))) for c, b in zip(r.boxes.cls, r.boxes.xyxy)]
    persons = [b for n, b in dets if n.lower() == 'person']
    others  = [(n, b) for n, b in dets if n.lower() != 'person']
    if not persons:
        persons = [[0, 0, img.shape[1], img.shape[0]]]
    for p in persons:
        near = [n for n, b in others if center_in(b, p)]
        has_vest = any(n == 'vest' for n in near)
        violations = [n for n in near if n in ('no_helmet', 'no_goggle')]
        if not has_vest: violations.append('no_vest')
        ok = len(violations) == 0
        color = (0,170,0) if ok else (0,0,220)
        label = 'COMPLIANT' if ok else 'NON-COMPLIANT: ' + ','.join(v.replace('no_','no ') for v in violations)
        x1,y1,x2,y2 = map(int, p)
        cv2.rectangle(img,(x1,y1),(x2,y2),color,3)
        cv2.putText(img,label,(x1,max(20,y1-8)),cv2.FONT_HERSHEY_SIMPLEX,0.6,color,2,cv2.LINE_AA)
    return img

test_dir = os.path.join(SETTINGS['datasets_dir'], 'construction-ppe', 'images', 'test')
imgs = sorted(glob.glob(os.path.join(test_dir, '*')))[:6]
os.makedirs('/content/compliance_out', exist_ok=True)
for ip in imgs:
    out = annotate(ip)
    op = os.path.join('/content/compliance_out', os.path.basename(ip))
    cv2.imwrite(op, out)
    display(Image(filename=op, width=560))
print('Saved', len(imgs), 'annotated images to /content/compliance_out')

## 7 - Zip + download all results

In [ ]:
import shutil, os
os.makedirs('/content/ppe_results', exist_ok=True)
shutil.copy(os.path.join(RUN_DIR,'weights','best.pt'), '/content/ppe_results/best.pt')
for f in ['results.png','results.csv','confusion_matrix.png','confusion_matrix_normalized.png','PR_curve.png','val_batch0_pred.jpg']:
    p = os.path.join(RUN_DIR, f)
    if os.path.exists(p): shutil.copy(p, '/content/ppe_results/')
shutil.copytree('/content/compliance_out','/content/ppe_results/compliance_out', dirs_exist_ok=True)
shutil.make_archive('/content/ppe_results','zip','/content/ppe_results')
print('Zipped. Downloading ppe_results.zip ...')
try:
    from google.colab import files; files.download('/content/ppe_results.zip')
except Exception as e:
    print('If the download does not start, grab it from the Files panel: /content/ppe_results.zip', e)

## Outputs
- `best.pt` - your trained YOLO11 PPE model
- `results.png`, `results.csv` - training curves + raw metrics
- `confusion_matrix.png`, `PR_curve.png` - evaluation plots
- `compliance_out/` - sample images labeled Compliant / Non-compliant

The trained weights and evaluation plots are saved under `docs/results/`.